In [1]:
import pandas as pd
import numpy as np

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(r"C:/Users/PC/Road-accident-blackspot-detection-Project/data/raw/ma3route_crashes_cleaned.csv")
df['crash_datetime'] = pd.to_datetime(df['crash_datetime'])
df['crash_date']     = pd.to_datetime(df['crash_date'])

In [2]:
# 1. DATASET OVERVIEW
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Total crash records   : {len(df):,}")
print(f"Total columns         : {df.shape[1]}")
print(f"Date range            : {df['crash_date'].min()} to {df['crash_date'].max()}")
print(f"Unique road segments  : {df['road_segment_id'].nunique():,}")
print(f"Missing values        : {df.isnull().sum().sum()}")
print(f"Duplicate rows        : {df.duplicated().sum()}")


DATASET OVERVIEW
Total crash records   : 31,064
Total columns         : 19
Date range            : 2012-08-08 00:00:00 to 2023-07-12 00:00:00
Unique road segments  : 1,693
Missing values        : 0
Duplicate rows        : 0


In [3]:
# 2. NUMERIC VARIABLES — CENTRAL TENDENCY & SPREAD
print("\n" + "=" * 60)
print("NUMERIC VARIABLES — DESCRIPTIVE STATISTICS")
print("=" * 60)

numeric_cols = ['n_crash_reports', 'severity_score', 'latitude',
                'longitude', 'hour_of_accident']

desc = df[numeric_cols].describe().T
desc['skewness'] = df[numeric_cols].skew()
desc['kurtosis'] = df[numeric_cols].kurt()
desc['median']   = df[numeric_cols].median()
desc['IQR']      = df[numeric_cols].quantile(0.75) - df[numeric_cols].quantile(0.25)
desc['CV%']      = (df[numeric_cols].std() / df[numeric_cols].mean() * 100).round(2)

print(desc[['count','mean','median','std','IQR','min','max',
            'skewness','kurtosis','CV%']].round(4).to_string())



NUMERIC VARIABLES — DESCRIPTIVE STATISTICS
                    count     mean   median     std      IQR     min      max  skewness  kurtosis     CV%
n_crash_reports   31064.0   1.4009   1.0000  1.4865   0.0000   1.000  66.0000   12.4937  298.4023  106.11
severity_score    31064.0   1.6598   1.0000  1.2116   1.0000   1.000  10.0000    2.3203    5.9568   73.00
latitude          31064.0  -1.2725  -1.2810  0.1190   0.0714  -3.100  -0.5654   -2.5412   38.7874   -9.35
longitude         31064.0  36.8525  36.8403  0.1137   0.0933  36.284  37.8795    2.2164   12.5123    0.31
hour_of_accident  31064.0  12.9357  13.0000  5.5251  10.0000   0.000  23.0000   -0.0507   -1.1187   42.71


In [4]:
# 3. CRASH REPORTS — TAIL ANALYSIS (key for GPD)
print("\n" + "=" * 60)
print("N_CRASH_REPORTS — TAIL ANALYSIS (GPD input)")
print("=" * 60)

cr = df['n_crash_reports']
percentiles = [50, 75, 90, 95, 99, 99.5]
for p in percentiles:
    print(f"  P{p:<5} : {np.percentile(cr, p):.1f}")

print(f"\n  Crashes == 1 (min severity) : {(cr == 1).sum():,}  ({(cr==1).mean()*100:.1f}%)")
print(f"  Crashes > 5                 : {(cr > 5).sum():,}  ({(cr>5).mean()*100:.1f}%)")
print(f"  Crashes > 10                : {(cr > 10).sum():,}  ({(cr>10).mean()*100:.1f}%)")
print(f"  Crashes > 20                : {(cr > 20).sum():,}  ({(cr>20).mean()*100:.1f}%)")
print(f"  Max (single event)          : {cr.max()}")


N_CRASH_REPORTS — TAIL ANALYSIS (GPD input)
  P50    : 1.0
  P75    : 1.0
  P90    : 2.0
  P95    : 3.0
  P99    : 7.0
  P99.5  : 9.0

  Crashes == 1 (min severity) : 25,252  (81.3%)
  Crashes > 5                 : 528  (1.7%)
  Crashes > 10                : 127  (0.4%)
  Crashes > 20                : 23  (0.1%)
  Max (single event)          : 66


In [5]:
# 4. SEVERITY SCORE DISTRIBUTION
print("\n" + "=" * 60)
print("SEVERITY SCORE DISTRIBUTION")
print("=" * 60)

sev_dist = df['severity_score'].value_counts().sort_index()
sev_pct  = (sev_dist / len(df) * 100).round(2)
sev_table = pd.DataFrame({'Count': sev_dist, 'Percentage': sev_pct})
print(sev_table.to_string())


SEVERITY SCORE DISTRIBUTION
                Count  Percentage
severity_score                   
1               20894       67.26
2                4964       15.98
3                2355        7.58
4                1648        5.31
5                 516        1.66
6                 435        1.40
7                 149        0.48
8                  81        0.26
9                  20        0.06
10                  2        0.01


In [6]:
# 5. BINARY / BOOLEAN FLAG COLUMNS
print("\n" + "=" * 60)
print("INCIDENT TYPE FLAGS — FREQUENCY & RATE")
print("=" * 60)

flag_cols = {
    'contains_fatality_words'  : 'Fatality',
    'contains_pedestrian_words': 'Pedestrian',
    'contains_matatu_words'    : 'Matatu',
    'contains_motorcycle_words': 'Motorcycle',
    'is_weekend'               : 'Weekend',
    'fatality_indicator'       : 'Fatality indicator',
    'pedestrian_involvement'   : 'Pedestrian involvement',
    'motorcycle_involvement'   : 'Motorcycle involvement',
}

for col, label in flag_cols.items():
    count = df[col].astype(int).sum()
    pct   = count / len(df) * 100
    print(f"  {label:<28}: {count:>6,}  ({pct:.2f}%)")


INCIDENT TYPE FLAGS — FREQUENCY & RATE
  Fatality                    :  2,284  (7.35%)
  Pedestrian                  :    944  (3.04%)
  Matatu                      :  2,541  (8.18%)
  Motorcycle                  :  1,142  (3.68%)
  Weekend                     :  7,549  (24.30%)
  Fatality indicator          :  2,284  (7.35%)
  Pedestrian involvement      :    944  (3.04%)
  Motorcycle involvement      :  1,142  (3.68%)


In [7]:
# 6. TEMPORAL DISTRIBUTIONS
print("\n" + "=" * 60)
print("CRASHES BY HOUR OF DAY")
print("=" * 60)
hour_dist = df['hour_of_accident'].value_counts().sort_index()
for h, c in hour_dist.items():
    bar = '█' * (c // 100)
    print(f"  {h:02d}:00  {c:>5,}  {bar}")

print("\n" + "=" * 60)
print("CRASHES BY DAY OF WEEK")
print("=" * 60)
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_dist  = df['day_of_week_name'].value_counts().reindex(day_order)
for day, count in day_dist.items():
    pct = count / len(df) * 100
    print(f"  {day:<12}: {count:>6,}  ({pct:.1f}%)")

print("\n" + "=" * 60)
print("CRASHES BY MONTH")
print("=" * 60)
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
month_dist = df['month_name'].value_counts().reindex(month_order)
for month, count in month_dist.items():
    pct = count / len(df) * 100
    print(f"  {month:<12}: {count:>6,}  ({pct:.1f}%)")



CRASHES BY HOUR OF DAY
  00:00    271  ██
  01:00    189  █
  02:00    146  █
  03:00    129  █
  04:00    161  █
  05:00    576  █████
  06:00  2,654  ██████████████████████████
  07:00  2,984  █████████████████████████████
  08:00  2,174  █████████████████████
  09:00  1,597  ███████████████
  10:00  1,374  █████████████
  11:00  1,298  ████████████
  12:00  1,239  ████████████
  13:00  1,292  ████████████
  14:00  1,384  █████████████
  15:00  1,428  ██████████████
  16:00  1,595  ███████████████
  17:00  2,321  ███████████████████████
  18:00  2,070  ████████████████████
  19:00  1,790  █████████████████
  20:00  1,749  █████████████████
  21:00  1,424  ██████████████
  22:00    779  ███████
  23:00    440  ████

CRASHES BY DAY OF WEEK
  Monday      :  4,586  (14.8%)
  Tuesday     :  4,730  (15.2%)
  Wednesday   :  4,604  (14.8%)
  Thursday    :  4,537  (14.6%)
  Friday      :  5,058  (16.3%)
  Saturday    :  4,395  (14.1%)
  Sunday      :  3,154  (10.2%)

CRASHES BY MONTH
  Janua

In [8]:
# 7. SPATIAL SUMMARY
print("\n" + "=" * 60)
print("SPATIAL SUMMARY")
print("=" * 60)
print(f"  Latitude  range : {df['latitude'].min():.4f}  to  {df['latitude'].max():.4f}")
print(f"  Longitude range : {df['longitude'].min():.4f}  to  {df['longitude'].max():.4f}")
print(f"  Unique segments : {df['road_segment_id'].nunique():,}")

# Crashes per segment
seg_counts = df.groupby('road_segment_id')['n_crash_reports'].sum()
print(f"\n  Crashes per segment:")
print(f"    Mean   : {seg_counts.mean():.2f}")
print(f"    Median : {seg_counts.median():.2f}")
print(f"    Std    : {seg_counts.std():.2f}")
print(f"    Max    : {seg_counts.max()} ({seg_counts.idxmax()})")
print(f"    Min    : {seg_counts.min()}")

print("\n  Top 10 highest-crash road segments:")
print(seg_counts.sort_values(ascending=False).head(10).to_string())


SPATIAL SUMMARY
  Latitude  range : -3.1000  to  -0.5654
  Longitude range : 36.2840  to  37.8795
  Unique segments : 1,693

  Crashes per segment:
    Mean   : 25.70
    Median : 4.00
    Std    : 78.24
    Max    : 1120 (SEG_-1.206_36.9135)
    Min    : 1

  Top 10 highest-crash road segments:
road_segment_id
SEG_-1.206_36.9135    1120
SEG_-1.26_36.8415     1021
SEG_-1.2195_36.891     952
SEG_-1.2465_36.864     937
SEG_-1.332_36.8685     754
SEG_-1.3275_36.846     614
SEG_-1.332_36.8865     588
SEG_-1.269_36.8325     543
SEG_-1.287_36.8235     533
SEG_-1.2915_36.828     443


In [9]:
# 8. CROSS-TABULATION — SEVERITY BY INCIDENT TYPE
print("\n" + "=" * 60)
print("MEAN SEVERITY SCORE BY INCIDENT TYPE")
print("=" * 60)

for col, label in [('contains_fatality_words','Fatality'),
                   ('contains_pedestrian_words','Pedestrian'),
                   ('contains_matatu_words','Matatu'),
                   ('contains_motorcycle_words','Motorcycle')]:
    mean_sev = df.groupby(col)['severity_score'].mean()
    print(f"\n  {label}:")
    for k, v in mean_sev.items():
        print(f"    {'Present' if k else 'Absent':<10}: mean severity = {v:.3f}")



MEAN SEVERITY SCORE BY INCIDENT TYPE

  Fatality:
    Absent    : mean severity = 1.399
    Present   : mean severity = 4.944

  Pedestrian:
    Absent    : mean severity = 1.572
    Present   : mean severity = 4.471

  Matatu:
    Absent    : mean severity = 1.549
    Present   : mean severity = 2.903

  Motorcycle:
    Absent    : mean severity = 1.610
    Present   : mean severity = 2.961


In [10]:
# 9. WEEKEND vs WEEKDAY COMPARISON
print("\n" + "=" * 60)
print("WEEKEND vs WEEKDAY")
print("=" * 60)

wk = df.groupby('is_weekend').agg(
    total_crashes     = ('n_crash_reports', 'sum'),
    crash_events      = ('crash_id', 'count'),
    mean_severity     = ('severity_score', 'mean'),
    fatality_rate     = ('contains_fatality_words', 'mean'),
    pedestrian_rate   = ('contains_pedestrian_words', 'mean'),
).rename(index={False: 'Weekday', True: 'Weekend'})
print(wk.round(4).to_string())


WEEKEND vs WEEKDAY
            total_crashes  crash_events  mean_severity  fatality_rate  pedestrian_rate
is_weekend                                                                            
Weekday             33626         23515         1.6629         0.0691           0.0301
Weekend              9892          7549         1.6502         0.0872           0.0313


In [11]:
# 10. EXPORT SUMMARY TABLE
summary = df[numeric_cols].describe().T
summary['skewness'] = df[numeric_cols].skew()
summary['kurtosis'] = df[numeric_cols].kurt()
summary.to_csv('descriptive_stats_summary.csv')
print("\n✓ Summary exported to descriptive_stats_summary.csv")


✓ Summary exported to descriptive_stats_summary.csv
